# 🚀 Evaluate the distilled shopping agent on SageMaker AI: ShoppingBench ASR and IFEval

Companion to `distill-tool-call--Qwen--Qwen3.5-4B-shopping.ipynb`. That notebook distills a teacher's shopping-agent
trajectories into Qwen3.5-4B with Sequence-level knowledge distillation (supervised fine-tuning on the teacher's outputs). This notebook checks whether
distillation helped, by evaluating three models as agents on the same held-out shopping tasks:

| Model | Served from |
|---|---|
| **Base**: Qwen3.5-4B before training | SageMaker AI real-time endpoint |
| **Student**: the distilled model | SageMaker AI real-time endpoint (deployed directly, or via ModelBuilder) |
| **Teacher / frontier reference** (optional): Claude Sonnet 4.6 | Amazon Bedrock (no endpoint to deploy) |

It reports two numbers per model:

- **ASR** (Absolute success rate) on ShoppingBench: Does the agent find a product that satisfies the shopper's request? Should go **up** for the student vs. base.
- **IFEval** (optional): General instruction following of the model.

The benchmark and how it is scored are explained next, before any code. The code lives in `scripts/`; the notebook
configures and calls it:

| Script | Role |
|---|---|
| `scripts/setup_shopbench.sh` | Install the benchmark harness, product catalog and search index (once) |
| `scripts/make_clean_split.py` | Keep only test problems the training data never saw |
| `scripts/sm_endpoints.py` | Deploy / delete the base and student endpoints |
| `scripts/sm_openai_relay.py` | Local OpenAI-compatible proxy that signs each request for SageMaker (or sends it to Bedrock) |
| `scripts/shopbench_harness.py` | Start the search server, run the agent trajectories, health checks |
| `scripts/score_rollouts.py` | Score trajectories with the benchmark's own rules |
| `scripts/run_ifeval.py` | IFEval with lm-evaluation-harness |

## What ShoppingBench measures

[ShoppingBench](https://github.com/yjwjy/ShoppingBench) ([paper](https://arxiv.org/abs/2508.04266)) gives an agent a
shopper's request and tools over "a large-scale shopping sandbox incorporating over 2.5 million real-world
products". The agent must search, inspect products and recommend the one(s) that meet every requirement. The paper
defines four user **intents** of increasing difficulty; the harness names them `product`, `shop`, `voucher` and `web`:

| Intent (paper name) | The shopper wants | Example |
|---|---|---|
| `product` (Products Finder) | One product meeting a set of constraints | "Women's jeans in size eu:30, baggy fit, high waist, denim, plain, above 114 PHP." |
| `shop` (Multi-products seller) | Several products, all bought from **one** shop | "Find shops offering gold jewelry that include: a necklace with zircon gemstones, a gold ring that's adjustable and priced between 208 and 924 PHP, and ..." |
| `voucher` (Coupon & Budget) | Several products whose total, after applying a voucher, fits a budget | "...a phone strap, a fuel pump filter, and an HP black ink bottle. My budget is only 398, but I have a voucher with the following rules: ..." |
| `web` (Knowledge) | A product identified through outside knowledge first | Not evaluated here (needs a web-search API) |

The test set has 250 problems for each of the first three intents and 150 for Knowledge; this notebook evaluates
only the held-out part of it (step 2). `EVAL_TASKS` below picks
the intents; `product` is the default.

**Tools the agent can call.** All run against a local search server over the catalog:

| Tool | Parameters | Returns |
|---|---|---|
| `find_product` | `q` (query), `page`; optional `price` range, `service` (`official`, `freeShipping`, `COD`, `flashsale`), `sort` (`priceasc`, `pricedesc`, `order`), `shop_id` | Up to 10 products (id, shop id, title, price, services, sold count) |
| `view_product_information` | `product_ids`: list of IDs e.g. `"4410867766,4422385859"` | Full details: attributes and SKU options (size, color, ...) |
| `recommend_product` | `product_ids` | The agent's answer. The **last** call in a trajectory is what gets scored; a JSON list instead of the string counts as no recommendation |
| `terminate` | `status` | Ends the trajectory |

**The agent loop (one trajectory).** At every step the model receives a system prompt with the tool descriptions and the corpus history so far, and replies in a fixed text format:

```
<think> reasoning ... </think>    (when thinking is on)
<tool_call>[{"name": "find_product", "parameters": {"q": "baggy high waist denim jeans", "page": 1}}]</tool_call>
```

The harness parses the `<tool_call>` block, runs the tools, appends their output to the history, and asks again. An
trajectory ends when the agent calls `terminate`, after completing it's maximum number of steps, or when the reply cannot be parsed (no
`<tool_call>` or `<response>` tag). That last case is a common failure of small models, and distillation fixes it.

## How a trajectory is scored (ASR)

Each test problem was generated from a real **gold product**. Along with the query, it stores the constraints that
product satisfies. An example:

```json
{"query": "Show me women's jeans in size eu:30 with a baggy fit, high waist, made of denim, plain design, and priced above 114 PHP.",
 "reward": {"product_id": "5113553158",
            "title": ["American Retro Baggy Jeans For Women High Waist Straight Denim ..."],
            "sku_options": [{"size": "eu:30"}],
            "attributes": [{"jeans_fit_type": ["baggy fit"]}, {"waist_type": ["high"]}, {"clothing_material": ["denim"]}, {"fa_pattern": ["plain"]}],
            "price": [{"greater than": [114, null]}]}}
```

The recommended product does **not** have to be the gold one. It is checked against every stored constraint:

| Check | Passes when |
|---|---|
| title | Semantic similarity between the recommended and gold titles is ≥ 0.5 (cosine, `Qwen3-Embedding-0.6B` embeddings), i.e. the same kind of product |
| price | Each bound holds: `less than`, `greater than`, or `between` |
| service | Every required service (e.g. `freeShipping`, `COD`) is offered |
| SKU options + attributes | Every required (key, value) pair, e.g. `size = eu:30`, `waist_type = high`, is found in the product's attributes or in one of its SKU variants (the best-matching variant counts) |

The share of checks passed is the product's **rule score**. Recommending the exact gold product scores 1 directly.
A trajectory is **successfully completed** when:

| Intent | Successfully completed if |
|---|---|
| `product` | The first recommended product passes every check (rule score = 1) |
| `shop` | Every requested product is recommended and passes every check, **and** all come from one shop |
| `voucher` | Every requested product passes every check, **and** the total after the voucher is within budget |

**Metrics reported**

| Metric | Meaning |
|---|---|
| `asr` | Absolute Success Rate: share of problems successfully completed (the headline number) |
| `exact_id` | Share where the exact gold product was recommended (stricter; many valid products exist) |
| `recommend` | Share of trajectories that ended with any recommendation at all. Most of the SeqKD lift shows up here: the base model often stops on an unparsable reply |

These rules are ShoppingBench's rule-based scorer: `src/agent/rewards/orm.py` ("outcome reward model"), applied by
`src/agent/run_evaluate.py`. `scripts/score_rollouts.py` calls those same functions.

## Evaluation parameters

| Parameter | Value | Reason |
|---|---|---|
| `ENABLE_THINKING` | `True` | Qwen3.5 reasons in `<think>` before each action. |
| `temperature` | 0 | Greedy decoding: repeatable runs |
| `max_tokens` | 8192 per step | Required for one reasoning block + tool call |
| `MAX_MODEL_LEN` | 40960 | The harness resends the whole history at every step, so prompts grow with trajectory length |
| Max steps | 30 | Fixed in the harness |
| `ROLLOUT_THREADS` | 4 per model | Concurrent trajectories per model; all models run at once, all sharing one search server |
| Search tool | 60 s timeout, 10 retries | Fixed in the harness; failures show up as `null_tool_results` in the health checks |
| Title similarity threshold | 0.5 | Fixed in the scorer |
| IFEval | thinking **off**, `max_tokens` 1280 | IFEval grades the whole reply, so a reasoning block would be scored against the format rules |

## 0. Environment (run once, then switch kernel)

The notebook's packages (like SageMaker SDK, OpenAI client) are installed in a virtual environment registered as a
Jupyter kernel. The ShoppingBench harness and lm-eval get their own
environments later (their pinned dependencies conflict).

Where it runs: the harness, search server and scoring run **on your machine** (CPU only),
e.g. a SageMaker Studio `ml.m5.4xlarge` space or a laptop. Only the models run on SageMaker endpoints.

Run the next cell in any kernel, then switch the notebook's kernel to **seqkd-eval** and continue from step 0b.

In [ ]:
import subprocess, sys
from pathlib import Path
WORK = Path.home() / "shopbench-eval"; WORK.mkdir(exist_ok=True)
EVAL_ENV = WORK / "evalenv"
if not (EVAL_ENV / "bin" / "python").exists():
    subprocess.run([sys.executable, "-m", "venv", str(EVAL_ENV)], check=True)
subprocess.run([str(EVAL_ENV / "bin/pip"), "install", "-q", "-U", "pip", "sagemaker>=3", "openai", "huggingface_hub",
                "transformers", "pandas", "matplotlib", "httpx", "requests", "boto3", "ipykernel"], check=True)
subprocess.run([str(EVAL_ENV / "bin/python"), "-m", "ipykernel", "install", "--user", "--name", "seqkd-eval",
                "--display-name", "seqkd-eval"], check=True)
print("Kernel 'seqkd-eval' registered. Switch to it (Kernel > Change Kernel), then run the cells below.")

## 0b. Configuration

**Permissions.** The execution role needs to create SageMaker models, endpoint configs and endpoints;
`sagemaker:InvokeEndpoint` on them; `sagemaker:CallWithBearerToken` (resource `"*"`); `bedrock:InvokeModel` for the
reference model; and read access to the Model Package's S3 bucket. Without the S3 access, the endpoint fails after ~2
minutes with only "Request to service failed".

In [ ]:
import os, json, sys, time, subprocess, logging
from pathlib import Path
assert "evalenv" in sys.executable, f"switch the kernel to 'seqkd-eval' first (current: {sys.executable})"
import boto3

REGION = boto3.Session().region_name or "us-west-2"
from sagemaker.core.helper.session_helper import get_execution_role
try:
    ROLE = get_execution_role()                 # inside SageMaker Studio / a notebook instance
except Exception:
    ROLE = "<arn:aws:iam::ACCOUNT:role/your-sagemaker-execution-role>"   # running elsewhere: set it explicitly

# ---- Models to evaluate ----
ENABLE_THINKING = False
BASE_HF = "Qwen/Qwen3.5-4B"
MODEL_PKG_ARN = "<model package ARN printed by step 03 of the training notebook>"
# Frontier reference row, run through the same harness via Bedrock. Set to None to skip.
# Its output is only scored here for comparison with frontier models.
REFERENCE_BEDROCK_MODEL = "us.anthropic.claude-sonnet-4-6"



EVAL_TASKS = ["product", "shop", "voucher"]          # held-out problems: product 103, shop 105, voucher 116
TRAIN_CORPUS = "oro-ai/sn15-shoppingbench-sft-15k"   
RUN_IFEVAL = True


# ---- endpoints (see scripts/sm_endpoints.py) ----

# Endpoints that are already InService are used as-is; only the missing models are deployed.
# (They must match SERVING: an endpoint created with the vLLM container needs SERVING = "vllm".)
REUSE_ENDPOINTS = {}   # e.g. {"base": "<endpoint name>"} to reuse an InService endpoint

# "vllm": SageMaker vLLM container. Starts on g6/g6e/p5; this cu130 build does not start on g5 / p4d hosts.
# "djl":  DJL/LMI container (vLLM engine inside). Starts on p4d (validated) and g5/g6e.
# Both honor the per-request thinking switch on the endpoint's OpenAI-compatible path.
SERVING = "vllm"
USE_MODELBUILDER = False   # deploy the student with the SDK's ModelBuilder instead (step 3); needs SERVING = "djl"
# Following istances are tried in order; on InsufficientInstanceCapacity or CannotStartContainerError the next one is used.
# p5 / p4d are 8-GPU fallbacks for a model that needs one GPU: delete them promptly (step 6).
INSTANCE_TYPES = {"djl": ["ml.g6e.xlarge", "ml.g6e.4xlarge", "ml.g5.2xlarge", "ml.p4d.24xlarge"],
                  "vllm": ["ml.g6e.xlarge", "ml.g6e.4xlarge", "ml.g6.2xlarge", "ml.g6e.2xlarge", "ml.p5.48xlarge"]}[SERVING]

MAX_MODEL_LEN = 40960

# ---- harness ----
ROLLOUT_THREADS = 12
WORK = Path.home() / "shopbench-eval"   # harness, index, venvs and outputs (~15 GB)
RUN_NAME = ""                           # e.g. "djl-p4d": keeps this run's rollouts/scores in their own folder
RELAY_PORT = 8101                       # first local relay port (change if another kernel already uses it)
OUT = WORK / "out" / RUN_NAME if RUN_NAME else WORK / "out"
OUT.mkdir(parents=True, exist_ok=True)
MAX_PROBLEMS = None #None=full run
SCRIPTS = Path("scripts").resolve()     # this notebook's folder holds scripts/
assert (SCRIPTS / "shopbench_harness.py").exists(), "run the notebook from its own folder (distillation/)"
sys.path.insert(0, str(SCRIPTS))

for name in ("botocore", "httpx", "httpx2"):               # quiet per-request logs
    logging.getLogger(name).setLevel(logging.WARNING)

def run(*args, **kw):
    """Run a command and stream its output into the notebook."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    for line in p.stdout: print(line, end="")
    if p.wait(): raise RuntimeError(f"exit code {p.returncode}: {args}")
print(REGION, ROLE)

## 1. Product search: Lucene index and search server (this machine, CPU)

The agent's tools never call a live shopping site. They query a **local copy of the product catalog** through a
search server, so every model sees exactly the same products and results. 


**The catalog.** `resources/documents.jsonl.gz` (1.4 GB) has one JSON record per product, 2,746,368 in total:

```json
{"id": "126564",
 "contents": "Rainbow Loom Refill (Navy Blue Opaque)\nnavy blue 5 10\nloom bands 6 years and above ...",
 "product": {"product_id": "126564", "shop_id": "7889", "title": "Rainbow Loom Refill (Navy Blue Opaque)",
             "price": 75.0, "sold_count": 1, "service": ["official"],
             "sku_options": {"1": {"color family": "navy blue", ...}},
             "attributes": {"recommended_age": ["6 years and above"], ...}, ...}}
```

`contents` is the **searchable text**: the title, the SKU option values and the attribute values. `product` is the
full record that the tools return and the scorer checks.

**The Lucene index.** [Pyserini](https://github.com/castorini/pyserini), a Python toolkit over the
[Lucene](https://lucene.apache.org/) search library (Java, hence the JDK), builds an inverted index of `contents`. It
maps each word, lower-cased and stemmed, to the products that contain it. It also stores each raw record, so a product
can be fetched by its `id`. The index is 3.0 GB and takes ~2 minutes to build with 8 threads on a laptop (longer on
smaller instances). A partial index left by an interrupted build is detected and rebuilt.

**The search server.** `src/search_engine/server.py` is a Flask app that loads the index and listens on
`127.0.0.1:5631`, the address hard-coded in the agent's tools. It runs in the background with one worker thread per
CPU core (at least 4), and its log is `<work_dir>/logs/search.log`. It has two routes, one per tool:

| Route (tool) | What it does |
|---|---|
| `/find_product` | 1. Ranks all products against `q` with **BM25**, the standard keyword-relevance score (rewards products containing the query's words, especially rarer ones, in shorter texts). 2. Walks down the ranking and drops products that fail the optional filters: `shop_id` (exact shop), `price` (`"low-high"`, e.g. `"100-500"`, `"1000-"`), `service` (each listed service must be offered). 3. Keeps the **first 50** that pass. 4. Optionally re-sorts those 50 by `sold_count` (`order`) or price (`priceasc` / `pricedesc`). 5. Returns page `page` (1-5) of 10, each with `product_id`, `shop_id`, `title`, `price`, `service`, `sold_count` |
| `/view_product_information` | Looks up each ID in `product_ids` directly (no ranking). Returns `short_description`, `description`, `sku_options` and `attributes`; unknown IDs are skipped |

What this means for the agent:
- Search is **keyword** matching, not semantic. Wording the query like the product titles matters, and so does
  retrying with other words.
- Sorting only reorders the 50 most relevant matches. `priceasc` gives the cheapest *relevant* product, not the
  cheapest in the catalog.
- Size, color and other variant details are only visible after `view_product_information`. That is why the
  SKU/attribute checks in the scorer reward agents that inspect products before recommending.

If trajectories later show many failed tool calls (`null_tool_results` in the health checks), the server is overloaded
or stuck: restart it with `Harness.start_search()`.

In [ ]:
#One-time setup for Shopping Bench Setup
run("bash", SCRIPTS / "setup_shopbench.sh", WORK, env={**os.environ, "PYTHON": sys.executable})

from shopbench_harness import Harness, nlines
harness = Harness(WORK)
shoppingBenchDir = harness.sb                                          # the ShoppingBench checkout
if not harness.search_ok(): harness.start_search()
print("search server up")

## 2. Held-out test problems

Public ShoppingBench trajectory sets can be built on the test queries themselves: 147 of the 250 product test queries
appear verbatim in `oro-ai/sn15-shoppingbench-sft-15k`. A model trained on them could score well by recall, so this
notebook evaluates **only held-out problems**: those whose query never appears in `TRAIN_CORPUS` (ignoring case and
punctuation). `make_clean_split.py` writes them to `data/synthesize_<task>_test_clean.jsonl` next to each official
test file.

In [ ]:
run(sys.executable, SCRIPTS / "make_clean_split.py", "--shopbench", shoppingBenchDir, "--corpus", TRAIN_CORPUS, "--tasks", *EVAL_TASKS)
CLEAN = {t: shoppingBenchDir / "data" / f"synthesize_{t}_test_clean.jsonl" for t in EVAL_TASKS}   # held-out problems
if MAX_PROBLEMS:                        # quick test: only the first N held-out problems of each intent
    for t in EVAL_TASKS:
        rows = open(CLEAN[t]).readlines()[:MAX_PROBLEMS]
        CLEAN[t] = CLEAN[t].with_name(f"synthesize_{t}_test_clean_first{MAX_PROBLEMS}.jsonl")
        CLEAN[t].write_text("".join(rows))
print({t: sum(1 for _ in open(p)) for t, p in CLEAN.items()})

## 3. Deploy the base and student endpoints

- **base**: downloaded from Hugging Face at container start.
- **student**: from the Model Package. Its S3 prefix holds the whole training output, so `student_model_data` picks
  the merged Hugging Face folder (`checkpoints/hf_merged/`), which is mounted at `/opt/ml/model`.

Both get one GPU, a 40k context and up to 16 concurrent sequences. Native tool-call parsing stays **off**: the
harness reads tool calls from the reply text, and a server-side tool parser would move them out of it. Deployment
takes about 10 minutes, and both deploy in parallel. Endpoints bill while in service; step 6 deletes them.

**Alternative: `ModelBuilder`** (`USE_MODELBUILDER = True`, `SERVING = "djl"`). The SageMaker SDK resolves the Model
Package's container image and merged weights. It also applies the base model's hosting recipe, which **overrides your
environment settings** (tensor parallel 4, eager mode; published only for 4-GPU g6/g6e). The script keeps
ModelBuilder's resolution and deploys it with one GPU, CUDA graphs on and the 40k context. On p4d it gave the same
ASR as the vLLM deployment.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from sm_endpoints import EndpointDeployer
DEP = EndpointDeployer(REGION, ROLE, serving=SERVING, instance_types=INSTANCE_TYPES, max_model_len=MAX_MODEL_LEN)

TO_DEPLOY = {"base": lambda: DEP.deploy("base", BASE_HF),
             "student": (lambda: DEP.deploy_student_with_modelbuilder(MODEL_PKG_ARN, INSTANCE_TYPES[-1]))
                        if USE_MODELBUILDER else
                        (lambda: DEP.deploy("student", "/opt/ml/model", DEP.student_model_data(MODEL_PKG_ARN)))}
ENDPOINTS = dict(REUSE_ENDPOINTS)

for ep in REUSE_ENDPOINTS.values():
    assert DEP.status(ep) == "InService", f"{ep} is not InService"
with ThreadPoolExecutor(2) as pool:
    futs = {tag: pool.submit(fn) for tag, fn in TO_DEPLOY.items() if tag not in ENDPOINTS}
    ENDPOINTS.update({tag: f.result() for tag, f in futs.items()})
print(ENDPOINTS)

### Connect the harness to the endpoints

The harness talks to models with an OpenAI client (fixed URL and API key). SageMaker endpoints instead require a
short-lived bearer token. `sm_openai_relay.start_relay` runs a small proxy on `127.0.0.1` per model that forwards each
request to the endpoint's OpenAI-compatible path (`/endpoints/<name>/openai/v1/chat/completions`) with a fresh
token, so multi-hour runs never fail on token expiry. For the reference model, the relay translates the request to
the Bedrock Converse API instead.

The smoke test also confirms the thinking switch reaches the server: with thinking on the reply starts with
reasoning (closed by `</think>`), with thinking off it does not. If both look the same, results would not reflect
`ENABLE_THINKING`.

In [ ]:
from openai import OpenAI
import sm_openai_relay as relay
relay.stop_all()                                   # re-running this cell: free the ports first
URLS = {tag: relay.start_relay(tag, ep, RELAY_PORT + i, REGION) for i, (tag, ep) in enumerate(ENDPOINTS.items())}
if REFERENCE_BEDROCK_MODEL:
    URLS["teacher"] = relay.start_relay("teacher", REFERENCE_BEDROCK_MODEL, RELAY_PORT + 98, REGION, transport="bedrock")

from shopbench_harness import has_reasoning
for tag, url in URLS.items():
    c = OpenAI(base_url=url, api_key="unused")
    ask = lambda **kw: c.chat.completions.create(model="", max_tokens=1024, messages=[{"role": "user", "content": "What is 17*3?"}], **kw).choices[0]
    if tag == "teacher":                      # Bedrock: no thinking switch; just check the relay answers
        print(f"{tag:8s} connected: {ask().message.content[-40:]!r}")
        continue
    for flag in (True, False):
        ch = ask(extra_body={"chat_template_kwargs": {"enable_thinking": flag}})
        out = ch.message.content
        reasoned = has_reasoning(out) or (ch.finish_reason == "length" and "</think>" not in out)  # cut off mid-thought
        print(f"{tag:8s} enable_thinking={flag!s:5s} reasoned: {reasoned!s:5s}  {out[-40:]!r}")
# Expected: reasoned True with enable_thinking=True and False with False, for base and student.

## 4. Run the agents and score them (ASR)

Each model plays every held-out problem of each intent through the unmodified ShoppingBench agent loop, with the
parameters from the table above. All models run at the same time. Expect roughly 30 minutes per intent (~100
problems) with 4 threads per model.

Interrupting this cell stops the trajectories. Re-running it resumes where it stopped: finished problems are kept.

In [ ]:
# import shutil; shutil.copy(WORK / "out" / "rollout_teacher_product.jsonl", OUT / "rollout_teacher_product.jsonl")
JOBS = {}
for tag, url in URLS.items():
    for t in EVAL_TASKS:
        cfg, rollout = harness.write_rollout_cfg(OUT, tag, t, CLEAN[t], url, ENABLE_THINKING, threads=ROLLOUT_THREADS)
        JOBS[f"{tag}_{t}"] = (cfg, rollout, nlines(CLEAN[t]))
harness.run_rollouts(JOBS)

In [ ]:
print(CLEAN)
print(OUT)

In [ ]:
import pandas as pd
RESULTS = {}
for tag in URLS:
    for t in EVAL_TASKS:
        rollout = JOBS[f"{tag}_{t}"][1]
        RESULTS[(t, tag)] = {**harness.score(t, rollout, CLEAN[t]), **{f"health:{k}": v for k, v in harness.health(rollout).items()}}
df = pd.DataFrame([{"task": t, "model": tag, **r} for (t, tag), r in RESULTS.items()]).set_index(["task", "model"]).sort_index()
for t in EVAL_TASKS:
    if (t, "base") in RESULTS and (t, "student") in RESULTS:
        b, s = RESULTS[(t, "base")]["asr"], RESULTS[(t, "student")]["asr"]
        line = f"{t}: ASR base {b:.3f} -> student {s:.3f} ({100 * (s - b):+.1f} pts)"
        if (t, "teacher") in RESULTS:
            te = RESULTS[(t, "teacher")]["asr"]
            line += f"; reference {te:.3f}" + (f"; student closes {(s - b) / (te - b):.0%} of the gap" if te > b else "")
        print(line)

# Overall: mean of the per-intent ASRs (the paper's "domain-weighted" average; here over the evaluated intents only,
# held-out problems, so not directly comparable to the paper's Avg. column, which also includes Knowledge).
avg = {tag: sum(RESULTS[(t, tag)]["asr"] for t in EVAL_TASKS) / len(EVAL_TASKS)
       for tag in URLS if all((t, tag) in RESULTS for t in EVAL_TASKS)}
if len(EVAL_TASKS) > 1:
    print(f"Avg. ASR ({', '.join(EVAL_TASKS)}; held-out): " + ", ".join(f"{tag} {a:.3f}" for tag, a in avg.items()))
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

LABELS = {"base": "base", "student": "student", "teacher": "reference (Bedrock)"}
models = [m for m in ("base", "student", "teacher") if any((t, m) in RESULTS for t in EVAL_TASKS)]
groups = EVAL_TASKS + (["Avg."] if len(EVAL_TASKS) > 1 else [])

def value(metric, t, m):
    if t == "Avg.":
        vals = [RESULTS[(x, m)][metric] for x in EVAL_TASKS if (x, m) in RESULTS]
        return sum(vals) / len(vals) if len(vals) == len(EVAL_TASKS) else np.nan
    return RESULTS[(t, m)][metric] if (t, m) in RESULTS else np.nan

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
panels = [("asr", "ASR (share of problems solved)"),
          ("recommend", "Share of trajectories that recommended a product")]
x, w = np.arange(len(groups)), 0.8 / len(models)
for ax, (metric, title) in zip(axes, panels):
    for i, m in enumerate(models):
        vals = [value(metric, t, m) for t in groups]
        bars = ax.bar(x + (i - (len(models) - 1) / 2) * w, vals, w, label=LABELS[m])
        ax.bar_label(bars, labels=["" if np.isnan(v) else f"{v:.2f}" for v in vals], fontsize=8, padding=2)
    ax.set_xticks(x, groups); ax.set_ylim(0, 1.05); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
axes[0].legend(loc="upper right")
fig.suptitle("Held-out ShoppingBench problems" + (" (thinking on)" if ENABLE_THINKING else " (thinking off)"))
fig.tight_layout()
plt.show()

**Reading the table**

- Compare base and student on **`asr`**; `n` is the number of held-out problems scored.
- `recommend` shows how often a trajectory ends with an answer at all. `exact_id` is a stricter view of the same result.
- Health columns (shares of trajectories) tell you whether the run itself was sound:
  - `empty_content`: the model call returned nothing (endpoint errors, context overflow). Should be ~0.
  - `null_tool_results`: a tool call failed, usually the search server timing out. Should be a few % at most; if
    higher, restart it with `harness.start_search()` and re-run step 4.
  - `no_terminate`: the trajectory ended without `terminate`, mostly on an unparsable reply. High for the base model;
    distillation lowers it.
  - `steps_with_reasoning`: ~1.0 with `ENABLE_THINKING = True`, ~0 with `False`.

Reference run (LoRA r64 student, held-out problems, thinking on, vLLM on p5):

| Model | product | shop | voucher | Avg. |
|---|---|---|---|---|
| base Qwen3.5-4B | 0.379 | 0.114 | 0.095 | 0.196 |
| SeqKD student | 0.466 | 0.343 | 0.129 | 0.313 |

Shop gains most: the student now finishes multi-product tasks (`no_terminate` 0.80 → 0.37). Expect a few points of
run-to-run variation.

## 5. IFEval (optional): general instruction following

[IFEval](https://arxiv.org/abs/2311.07911) asks for responses with verifiable format rules ("answer in exactly 3
bullet points", "no commas", ...). It checks that training on shopping trajectories did not erode general instruction
following. **prompt_strict** is the share of prompts where every rule is followed.

IFEval grades the whole reply, so it runs with thinking **off**. lm-eval cannot set `chat_template_kwargs` itself, so
a second relay per model forces `{"enable_thinking": false}` into every request. `run_ifeval.py` then counts
predictions that still contain reasoning (`reasoning_leaks`); any leak makes the score invalid.

Compare base and student: a small drop is the expected cost of specializing; full fine-tuning tends to drop much
more, which is why the training notebook uses LoRA.

In [ ]:
if RUN_IFEVAL:
    LMEVAL_ENV = WORK / "lmevalenv"
    if not (LMEVAL_ENV / "bin/lm_eval").exists():
        run(sys.executable, "-m", "venv", LMEVAL_ENV)
        run(LMEVAL_ENV / "bin/pip", "install", "-q", "lm_eval[api,ifeval]")
    IFEVAL = {}
    for i, (tag, ep) in enumerate(ENDPOINTS.items()):
        url = relay.start_relay(f"ifeval_{tag}", ep, RELAY_PORT + 100 + i, REGION,
                                force={"chat_template_kwargs": {"enable_thinking": False}})
        out = subprocess.run([sys.executable, SCRIPTS / "run_ifeval.py", "--base-url", url, "--out", OUT / f"ifeval_{tag}",
                              "--lm-eval", LMEVAL_ENV / "bin/lm_eval"], capture_output=True, text=True)
        if out.returncode: raise RuntimeError(out.stderr[-3000:])
        IFEVAL[tag] = json.loads(out.stdout.strip().splitlines()[-1])
        print(tag, IFEVAL[tag], "" if IFEVAL[tag]["reasoning_leaks"] == 0 else "<- INVALID: reasoning leaked into predictions")

## 6. Delete the endpoints

Real-time endpoints bill while in service. This deletes every endpoint in `ENDPOINTS`, **including reused ones**:
remove an entry from `ENDPOINTS` first to keep it. The search server and relays stop here too (or with the kernel).

In [ ]:
for ep in ENDPOINTS.values():
    DEP.delete(ep, wait=False); print("deleted", ep)
harness.stop_all()
relay.stop_all()